# Project AEON: From-Scratch JEPA Crafter World Model & Agent (Colab GPU Runner)

This notebook trains and validates the **AEON JEPA** world model on raw Crafter visual interaction (`64×64×3` RGB).

### Non-Negotiable Integrity Invariants:
1. **Zero Pretrained Weights:** 100% learned parameters initialized randomly.
2. **Zero Game Knowledge:** No crafting recipes, no inventory requirements, no object identities, no semantic action descriptions.
3. **Anonymous Action Space:** The agent sees only anonymous tokens `0..16`, translated via an opaque permutation.
4. **Information Quarantine:** Crafter `info` dictionaries are completely stripped from the learner and routed strictly to metrics evaluation.
5. **Persistent Drive Storage:** Experience, replay buffer, and model checkpoints stream directly to `/content/drive/MyDrive/learn-a-world/aeon/` to survive Colab disconnections.

## 1. Mount Google Drive and Setup Dedicated AEON Subfolder

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Configure dedicated AEON model directory under learn-a-world
MASTER_DRIVE_DIR = Path('/content/drive/MyDrive/learn-a-world')
AEON_DRIVE_DIR = MASTER_DRIVE_DIR / 'aeon'

# Subdirectories for checkpoints, run logs, artifacts, and evaluation reports
for sub in ['checkpoints', 'runs', 'artifacts', 'results']:
    (AEON_DRIVE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Google Drive mounted successfully.")
print(f"Master Drive root: {MASTER_DRIVE_DIR}")
print(f"AEON dedicated folder: {AEON_DRIVE_DIR}")

## 2. Sync Repository from GitHub

In [ ]:
%cd /content
import os

# Clone or pull latest main branch
if not os.path.exists("learn-a-world"):
    !git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world
else:
    %cd /content/learn-a-world
    !git fetch origin
    !git pull origin main

%cd /content/learn-a-world
!git log -1 --oneline

## 3. Install Dependencies & Hardware Audit

In [ ]:
# Install lightweight requirements
!pip install --quiet -r AEON/jepa-crafter/requirements.txt

# Set PYTHONPATH to include AEON/jepa-crafter
import sys
aeon_src_dir = '/content/learn-a-world/AEON/jepa-crafter'
if aeon_src_dir not in sys.path:
    sys.path.insert(0, aeon_src_dir)

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 4. Run Architectural & Information Integrity Test Suite

Runs all 10 acceptance tests verifying:
- 64x64 uint8 RGB observation contract
- Zero information leakage from Crafter internals
- Exact sequence alignment across horizons 1, 2, 4, 8
- Stop-gradient EMA target encoder behavior
- Cross-Entropy Method latent replanning

In [ ]:
%cd /content/learn-a-world/AEON/jepa-crafter
!python3 -m pytest -v tests/

## 5. Launch Training Campaign with Direct Google Drive Persistence

Trains the full AEON agent system:
- Initial random warm-up
- Multi-horizon JEPA future latent prediction with EMA target network
- Automated validation gates (Test B heldout prediction, Test C action-conditioning, Test D collapse checks)
- Staged curiosity exploration and Cross-Entropy Method (CEM) latent model-predictive control (MPC)
- All checkpoints and logs save atomically to `/content/drive/MyDrive/learn-a-world/aeon/`

In [ ]:
# Execute training run saving directly to Google Drive (Phase 4: 50,000 steps pilot)
%cd /content/learn-a-world
!python3 AEON/jepa-crafter/run.py --config AEON/jepa-crafter/configs/crafter_t4.yaml --drive-root /content/drive/MyDrive/learn-a-world/aeon --max-steps 50000 --num-envs 8 --device auto


## 6. Inspect Results and Benchmark Scorecard

Loads the generated `final_report.md` and displays the official Crafter 22-achievement scorecard.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

aeon_run_dir = Path("/content/drive/MyDrive/learn-a-world/aeon/runs/jepa_crafter")

# 1. Display Final Research Report
report_path = aeon_run_dir / "final_report.md"
if report_path.exists():
    display(Markdown(report_path.read_text()))
else:
    print(f"Report not yet generated at {report_path}")

# 2. Display Achievement Rates Table
csv_path = aeon_run_dir / "achievement_rates.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("\n=== OFFICIAL CRAFTER BENCHMARK SCORECARD ===")
    display(df)


## 7. Visual "Imagination" Inspector (Real Scene Prediction Probe)

Tests the model's visual eyes and imagination on a real Crafter scene (trees, stone, terrain):
- **Panel 1**: Current observation $.
- **Panel 2**: Real next observation {t+1}$ after executing an action.
- **Panel 3**: Ground-truth pixel motion heatmap.
- **Panel 4**: JEPA latent prediction heatmap ( \times 8$ spatial patches upsampled to  \times 64$) overlaid on the scene.

Shows exactly which trees, stones, and terrain patches the model successfully predicted!


In [ ]:
# ==============================================================================
# Cell 13: Visual "Imagination" Inspector (Real Scene Prediction Probe)
# ==============================================================================
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from src.env.crafter_adapter import CrafterAdapter
from src.env.opaque_actions import OpaqueActionMapping
from src.models.jepa_world_model import JepaWorldModel
from src.data.checkpoint_store import CheckpointStore

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Load trained checkpoint from Google Drive (or local fallback)
model = JepaWorldModel().to(device)
model.eval()

candidate_dirs = [
    '/content/drive/MyDrive/learn-a-world/aeon/runs/jepa_crafter/checkpoints',
    '/content/drive/MyDrive/learn-a-world/aeon/checkpoints',
    'runs/jepa_crafter/checkpoints',
    'checkpoints'
]
loaded_ckpt = False
for c_dir in candidate_dirs:
    latest_pt = os.path.join(c_dir, 'checkpoint_latest.pt')
    if os.path.exists(latest_pt):
        ckpt = torch.load(latest_pt, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        if 'ema_state_dict' in ckpt:
            model.target_encoder.load_state_dict(ckpt['ema_state_dict'])
        print(f'[Loaded Checkpoint] Successfully loaded model from {latest_pt} (Step: {ckpt.get("step", "unknown")})')
        loaded_ckpt = True
        break

if not loaded_ckpt:
    print('[Notice] No saved checkpoint found yet. Running probe with initialized model architecture.')

# 2. Reset a rich Crafter scene (Seed 42 spawns with trees, stones, grass, and water nearby)
adapter = CrafterAdapter(seed=42)
obs_t = adapter.reset()

# 3. Test an Action (Action 1 = Step forward toward trees/stone)
test_action = 1
next_obs, reward, term, trunc = adapter.step(test_action)

# 4. Run JEPA Perception and Prediction
obs_ch = np.transpose(obs_t, (2, 0, 1))
obs_tensor = torch.from_numpy(obs_ch).float().div_(255.0).unsqueeze(0).unsqueeze(0).to(device) # [1, 1, 3, 64, 64]
act_tensor = torch.zeros(1, 1, dtype=torch.long, device=device)
mask_tensor = torch.ones(1, 1, dtype=torch.bool, device=device)

with torch.no_grad():
    # Eyes: Encode history
    z_t, h_t = model.encode_actual_history(obs_tensor, act_tensor, mask_tensor)
    
    # Imagination: Predict future spatial tokens conditioned on action
    a_emb = model.action_embedding(torch.tensor([test_action], device=device))
    z_hat = model.predictor_ensemble.forward_member(0, z_t, h_t, a_emb) # [1, 64, 128]
    
    # Target: Real next observation target tokens
    next_ch = np.transpose(next_obs, (2, 0, 1))
    next_tensor = torch.from_numpy(next_ch).float().div_(255.0).unsqueeze(0).to(device)
    z_target, _ = model.target_encoder(next_tensor) # [1, 64, 128]
    
    # Compare patch-by-patch (64 spatial tokens = 8x8 image patches)
    p_norm = F.normalize(z_hat, dim=-1)
    t_norm = F.normalize(z_target, dim=-1)
    patch_cosine = (p_norm * t_norm).sum(dim=-1).squeeze(0) # [64]
    
    # Persistence baseline for comparison
    curr_norm = F.normalize(z_t, dim=-1)
    persist_cosine = (curr_norm * t_norm).sum(dim=-1).squeeze(0)
    
    # Upscale 8x8 spatial grid to 64x64 for visualization
    grid_8x8 = patch_cosine.reshape(8, 8).unsqueeze(0).unsqueeze(0)
    heatmap_64 = F.interpolate(grid_8x8, size=(64, 64), mode='bicubic', align_corners=False).squeeze().cpu().numpy()

# 5. Render 4-Panel Qualitative Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Panel 1: Current Observation
axes[0].imshow(obs_t)
axes[0].set_title('1. Current Observation O_t\n(Player with Trees & Stone)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Panel 2: Real Next Frame
axes[1].imshow(next_obs)
axes[1].set_title(f'2. Real Next Frame O_{{t+1}}\n(After Action Token: {test_action})', fontsize=12, fontweight='bold')
axes[1].axis('off')

# Panel 3: Real Pixel Motion Difference
pixel_diff = np.abs(next_obs.astype(float) - obs_t.astype(float)).mean(axis=-1)
axes[2].imshow(pixel_diff, cmap='inferno')
axes[2].set_title('3. Real World Motion\n(Where pixels actually changed)', fontsize=12, fontweight='bold')
axes[2].axis('off')

# Panel 4: Spatial Prediction Accuracy Heatmap
axes[3].imshow(next_obs)
im = axes[3].imshow(heatmap_64, cmap='viridis', alpha=0.60)
axes[3].set_title(f'4. JEPA Prediction Heatmap\n(Mean Sim: {float(patch_cosine.mean()):.3f} vs Persist: {float(persist_cosine.mean()):.3f})', fontsize=12, fontweight='bold')
axes[3].axis('off')
cbar = fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)
cbar.set_label('Latent Accuracy (Cosine Sim)', rotation=270, labelpad=15)

plt.tight_layout()
plt.show()

# 6. Quantitative Verdict
mean_pred_sim = float(patch_cosine.mean())
mean_pers_sim = float(persist_cosine.mean())
print('=' * 65)
print('                   JEPA SCENE PREDICTION VERDICT')
print('=' * 65)
print(f'• Model Prediction Cosine Similarity : {mean_pred_sim:.4f}  (Higher is better)')
print(f'• Static Persistence Baseline        : {mean_pers_sim:.4f}')
if mean_pred_sim > mean_pers_sim:
    gain = ((mean_pred_sim - mean_pers_sim) / max(1e-6, mean_pers_sim)) * 100.0
    print(f'>>> RESULT: PASSED! Imagination beat static persistence by +{gain:.1f}% on this scene! <<<')
else:
    gap = ((mean_pers_sim - mean_pred_sim) / max(1e-6, mean_pers_sim)) * 100.0
    print(f'>>> RESULT: Persistence currently leads by {gap:.1f}%. Model needs further training epochs. <<<')
print('=' * 65)
